# Municipal technical-capacity tier — derivation and validation

Builds the four-tier capacity classification for all 345 Chilean comunas from INE Censo 2024
population and the two documented MEED rules. The census table is the only dataset read.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "dataset-review").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "dataset-review").exists(), "run from inside the CityCatalyst-global-data repository"

REVIEWS = ROOT / "dataset-review" / "reviews"
HERE = REVIEWS / "oef" / "cl-municipal-capacity-tier" / "releases" / "v1"
censo = pd.read_csv(
    REVIEWS / "cl-ine/cl-ine-censo/releases/2024/data/raw_data_cl_ine_censo.csv")
print(f"census rows {len(censo)}")


## Population per comuna

The census release is long-format. Keep the population attribute and preserve CUT as a five-character
string because leading zeros are significant.


In [ ]:
pop = (censo.loc[censo["attribute_type"] == "population",
                 ["comuna", "comuna_nombre", "attribute_value"]]
       .rename(columns={"comuna": "cut_int", "comuna_nombre": "comuna",
                        "attribute_value": "population"}))
pop["comuna_cut"] = pop["cut_int"].astype(int).astype(str).str.zfill(5)
pop["population"] = pd.to_numeric(pop["population"], errors="raise").astype(int)
pop = pop[["comuna_cut", "comuna", "population"]].sort_values("comuna_cut").reset_index(drop=True)


## Apply the population tramo and capacity bands

T1 is strictly above 100,000; T2 starts at 20,000; T3 starts at 5,000; T4 is below 5,000.
The T4 capacity value remains provisional because the source method says only that it is below 25.


In [ ]:
def vem_tramo(population):
    if population > 100_000:
        return "T1"
    if population >= 20_000:
        return "T2"
    if population >= 5_000:
        return "T3"
    return "T4"

TRAMO_LABEL = {
    "T1": "Grandes urbanas",
    "T2": "Medianas urbanas",
    "T3": "Pequeñas semiurbanas",
    "T4": "Rurales y aisladas",
}
GL_SCORE = {"T1": 100, "T2": 50, "T3": 25, "T4": 10}

pop["tramo"] = pop["population"].map(vem_tramo)
pop["tramo_label"] = pop["tramo"].map(TRAMO_LABEL)
pop["gl_technical_capacity"] = pop["tramo"].map(GL_SCORE)
pop["capacity"] = pop["gl_technical_capacity"] / 100
pop["capacity_basis"] = "vem_tramo_ine_censo_2024"
pop["t4_provisional"] = pop["tramo"].eq("T4")


## Validate and export


In [ ]:
assert len(pop) == 345, f"expected 345 comunas, got {len(pop)}"
assert pop["comuna_cut"].is_unique, "duplicate CUT"
assert pop["comuna_cut"].str.fullmatch(r"\d{5}").all(), "invalid CUT"
assert pop[["population", "tramo", "capacity"]].notna().all().all()
assert set(pop["capacity"]) == {1.0, 0.5, 0.25, 0.1}

bounds = {
    "T1": lambda s: s.gt(100_000),
    "T2": lambda s: s.ge(20_000) & s.le(100_000),
    "T3": lambda s: s.ge(5_000) & s.lt(20_000),
    "T4": lambda s: s.lt(5_000),
}
for tramo, check in bounds.items():
    values = pop.loc[pop["tramo"] == tramo, "population"]
    assert check(values).all(), f"{tramo}: population outside bracket"

OUT_COLS = [
    "comuna_cut", "comuna", "population", "tramo", "tramo_label",
    "gl_technical_capacity", "capacity", "capacity_basis", "t4_provisional",
]
out = pop[OUT_COLS].sort_values("comuna_cut").reset_index(drop=True)
out.to_csv(HERE / "data/municipal_capacity_tier.csv", index=False)
print(f"wrote data/municipal_capacity_tier.csv — {out.shape[0]} rows x {out.shape[1]} columns")
print(out["tramo"].value_counts().sort_index())


## Result

The release is derived from one explicitly licensed INE input. It contains 57 T1, 113 T2, 133 T3
and 42 T4 comunas. The T4 score and T1 evidence condition remain the two promotion decisions.
